# Streaming Covariance Observables Across Multiple Geometries

This Colab/A100 notebook runs canonical GPU Markov dynamics through `classA_U1FGTN_gpu.run_markov_circuit(...)` and computes streamed observables at every sample and every cycle without saving raw covariance data.

Saved per-run products:
- `scalar_metrics.csv`
- `entropy_y0avg_vs_ay.npz`
- `xavg_square_correlator_vs_ry.npz`
- `local_chern_marker.npz`
- `local_charge_cell.npz`
- `measurement_frustration.npz` for `perfect_correction` runs only
- `run_summary.json`
- `latest_run.json`

Campaign-level products:
- `campaign_manifest.json`
- `run_index.csv`
- `latest_campaign.json`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'


def is_bundle_root(path: Path) -> bool:
    return (
        (path / 'src' / 'classA_U1FGTN_gpu.py').exists()
        and (path / 'src' / 'streaming_covariance_observables_gpu.py').exists()
    )


candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with required src files')

PROJECT_ROOT = BUNDLE_ROOT.parent
BUNDLE_SRC_DIR = BUNDLE_ROOT / 'src'
PROJECT_SRC_DIR = PROJECT_ROOT / 'src'
for path in [BUNDLE_SRC_DIR, PROJECT_SRC_DIR]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] project root: {PROJECT_ROOT}')


In [ ]:
import os
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm
import torch

from fgtn.classA_U1FGTN import classA_U1FGTN
from classA_U1FGTN_gpu import classA_U1FGTN_gpu
from streaming_covariance_observables_gpu import (
    HELPER_VERSION,
    PROTOCOL_ORDER,
    StreamingCovarianceObservables,
    case_output_dir,
    config_id,
    domain_wall_metadata,
    expected_samples_for_config,
    geometry_key,
    rel_to_root,
    run_internal_checks,
    save_dataframe_atomic_csv,
    save_npz_atomic,
    write_json_atomic,
)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[setup] helper={HELPER_VERSION}')
print(f'[setup] cpu reference class: {classA_U1FGTN.__name__}')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
print(f'[setup] total gpu memory: {total_gib:.2f} GiB')
if total_gib < 35.0:
    print('[warn] This notebook is sized for an A100 40GB runtime.')

run_internal_checks()
print('[setup] helper internal checks passed')


## Campaign Configuration

Requested `post_selection` is stored internally as repo protocol `postselect`. The canonical GPU driver still forces `postselect` to one actual sample.


In [ ]:
GEOMETRIES = [
    {'Nx': 20, 'Ny': 20},
    {'Nx': 20, 'Ny': 30},
    {'Nx': 20, 'Ny': 40},
    {'Nx': 20, 'Ny': 50},
    {'Nx': 30, 'Ny': 30},
    {'Nx': 30, 'Ny': 40},
    {'Nx': 30, 'Ny': 50},
    {'Nx': 40, 'Ny': 40},
    {'Nx': 40, 'Ny': 50},
    {'Nx': 50, 'Ny': 50},
]
NSHELL = 1
INIT_MODE = 'default'
REQUESTED_SAMPLES = 100
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
TRIAL_ORBITALS = 'X'
N_A = 0.5
AUTOTUNE_ENTROPY = True
ENTROPY_AUTOTUNE_REPEAT = 1
SAMPLE_CHUNK_CANDIDATES = (1, 2, 4, 8, 16)
Y0_CHUNK_CANDIDATES = (1, 2, 4, 6, 8, 10, 12, 16, 20, 24, 32, 'ny')
MEMORY_SAFETY_FRACTION = 0.85
EIGH_MEMORY_MULTIPLIER = 8.0
OVERWRITE = True

PROTOCOL_CONFIGS = [
    {
        'protocol': 'perfect_correction',
        'protocol_display': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples': REQUESTED_SAMPLES,
        'batch_size': None,
    },
    {
        'protocol': 'postselect',
        'protocol_display': 'post_selection',
        'postselect': True,
        'perfect_correction': False,
        'samples': REQUESTED_SAMPLES,
        'batch_size': 1,
    },
]

CONFIGS = [
    {
        **geometry,
        'cycles': int(geometry['Ny']),
        'nshell': NSHELL,
        'init_mode': INIT_MODE,
        **protocol_cfg,
    }
    for geometry in GEOMETRIES
    for protocol_cfg in PROTOCOL_CONFIGS
]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'streaming_covariance_observables'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ID = 'multi_geometry_nsh1_dwtrunc1_init-default_S100_cycles-Ny'
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
BASE_SAVED_OBSERVABLES = [
    'frob_norm',
    'frob_successive_delta',
    'entropy_y0avg_vs_ay',
    'real_space_chern',
    'local_chern_marker',
    'xavg_square_correlator_vs_ry',
    'local_charge_cell',
]
PERFECT_CORRECTION_ONLY_OBSERVABLES = [
    's_Ap',
    's_Am',
    's_Bp',
    's_Bm',
    'frustration_min',
    'frustration_avg',
    'frustration_prod',
]
SAVED_OBSERVABLES_BY_PROTOCOL = {
    'perfect_correction': [*BASE_SAVED_OBSERVABLES, *PERFECT_CORRECTION_ONLY_OBSERVABLES],
    'postselect': list(BASE_SAVED_OBSERVABLES),
}


def saved_observables_for_config(cfg: dict[str, Any]) -> list[str]:
    return list(SAVED_OBSERVABLES_BY_PROTOCOL[str(cfg['protocol'])])


def observable_collection_mode_for_config(cfg: dict[str, Any]) -> str:
    if bool(cfg['perfect_correction']):
        return 'cycle_observer_and_site_observer_streaming'
    return 'cycle_observer_streaming'


def estimate_run_storage_bytes(cfg: dict[str, Any]) -> int:
    nx = int(cfg['Nx'])
    ny = int(cfg['Ny'])
    cycles = int(cfg['cycles'])
    samples_actual = expected_samples_for_config(cfg)
    scalars = samples_actual * cycles * 3
    curves = samples_actual * cycles * 2 * (ny // 2 + 1)
    field_count = 2 + (7 if bool(cfg['perfect_correction']) else 0)
    fields = samples_actual * cycles * field_count * nx * ny
    return int(8 * (scalars + curves + fields))


config_rows = []
for cfg in CONFIGS:
    config_rows.append({
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': int(cfg['cycles']),
        'nshell': int(cfg['nshell']),
        'protocol_display': cfg['protocol_display'],
        'protocol_saved': cfg['protocol'],
        'requested_samples': int(cfg['samples']),
        'actual_samples': expected_samples_for_config(cfg),
        'batch_size': cfg['batch_size'],
        'estimated_storage_gib_float64': estimate_run_storage_bytes(cfg) / 1024**3,
    })

config_df = pd.DataFrame(config_rows)
display(Markdown('### Campaign config'))
display(config_df)
print(json.dumps({
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'configs': CONFIGS,
    'covariance_history_saved': False,
    'estimated_storage_gib_float64': float(config_df['estimated_storage_gib_float64'].sum()),
    'protocol_note': 'postselect=True requests 100 samples but the canonical GPU driver forces samples=1.',
}, indent=2))


In [ ]:
def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        payload = json.loads(index_path.read_text())
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    campaigns[:] = [item for item in campaigns if item.get('name') != CAMPAIGN_NAME]
    campaigns.append({
        'name': CAMPAIGN_NAME,
        'kind': 'streamed_covariance_observables',
        'manifest': f'{CAMPAIGN_NAME}/latest_campaign.json',
    })
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)


def assert_overwrite_allowed(path: Path) -> None:
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f'Path already exists and OVERWRITE=False: {path}')


def run_config_payload(cfg: dict[str, Any], *, actual_samples: int, dw_meta: dict[str, Any]) -> dict[str, Any]:
    return {
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': int(cfg['cycles']),
        'nshell': int(cfg['nshell']),
        'init_mode': str(cfg['init_mode']),
        'protocol': str(cfg['protocol']),
        'protocol_display': str(cfg['protocol_display']),
        'postselect': bool(cfg['postselect']),
        'perfect_correction': bool(cfg['perfect_correction']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': int(actual_samples),
        'alpha_1': float(ALPHA_1),
        'alpha_2': float(ALPHA_2),
        'dw_truncation': bool(DW_TRUNCATION),
        'dtype': DTYPE,
        'device': DEVICE,
        'backend': BACKEND,
        'sequence': SEQUENCE,
        'batch_size_requested': cfg['batch_size'],
        'n_a': float(N_A),
        'trial_orbitals': TRIAL_ORBITALS,
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'covariance_history_saved': False,
        'observable_collection_mode': observable_collection_mode_for_config(cfg),
        'saved_observables': saved_observables_for_config(cfg),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }


def file_size_bytes(path: Path) -> int:
    return int(path.stat().st_size)


In [ ]:
def run_one_config(cfg: dict[str, Any]) -> dict[str, Any]:
    out_dir = case_output_dir(RUNS_ROOT, cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    expected_samples = expected_samples_for_config(cfg)
    protocol = str(cfg['protocol'])
    cycles = int(cfg['cycles'])
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} init={cfg['init_mode']} "
        f"protocol={protocol} requested_samples={cfg['samples']} actual_samples={expected_samples} "
        f"cycles={cycles} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    dw_meta = domain_wall_metadata(model)
    observer = StreamingCovarianceObservables(
        nx=int(cfg['Nx']),
        ny=int(cfg['Ny']),
        cycles=cycles,
        samples_expected=expected_samples,
        protocol=protocol,
        nshell=int(cfg['nshell']),
        dw_loc=dw_meta['dw_loc'],
        autotune_entropy=AUTOTUNE_ENTROPY,
        sample_chunk_candidates=SAMPLE_CHUNK_CANDIDATES,
        y0_chunk_candidates=Y0_CHUNK_CANDIDATES,
        autotune_repeat=ENTROPY_AUTOTUNE_REPEAT,
        memory_safety_fraction=MEMORY_SAFETY_FRACTION,
        eigh_memory_multiplier=EIGH_MEMORY_MULTIPLIER,
    )

    observer_pbar = tqdm(
        total=expected_samples * cycles,
        desc=f"observer {config_id(cfg['Nx'], cfg['Ny'], protocol)}",
        unit='sample-cycle',
        leave=False,
    )
    cycle_observer = observer.make_cycle_observer(progress_bar=observer_pbar)
    site_observer = observer.make_site_observer() if bool(cfg['perfect_correction']) else None
    try:
        result = model.run_markov_circuit(
            G_history=False,
            progress=False,
            cycles=cycles,
            postselect=bool(cfg['postselect']),
            perfect_correction=bool(cfg['perfect_correction']),
            samples=int(cfg['samples']),
            init_mode=str(cfg['init_mode']),
            save=False,
            save_init=False,
            save_history_stride=None,
            n_a=N_A,
            sequence=SEQUENCE,
            batch_size=cfg['batch_size'],
            return_data=False,
            cycle_observer=cycle_observer,
            site_observer=site_observer,
        )
    finally:
        observer_pbar.close()

    actual_samples = int(result['samples'])
    if actual_samples != expected_samples:
        raise RuntimeError(f'Expected {expected_samples} samples but run returned {actual_samples}.')
    observer.finalize(actual_samples=actual_samples)

    metrics_df = observer.metrics_dataframe().sort_values(['sample_index', 'cycle_label']).reset_index(drop=True)
    run_payload = run_config_payload(cfg, actual_samples=actual_samples, dw_meta=dw_meta)

    metrics_path = out_dir / 'scalar_metrics.csv'
    entropy_path = out_dir / 'entropy_y0avg_vs_ay.npz'
    corr_path = out_dir / 'xavg_square_correlator_vs_ry.npz'
    marker_path = out_dir / 'local_chern_marker.npz'
    charge_path = out_dir / 'local_charge_cell.npz'
    frustration_path = out_dir / 'measurement_frustration.npz' if bool(cfg['perfect_correction']) else None
    summary_path = out_dir / 'run_summary.json'
    latest_path = out_dir / 'latest_run.json'
    paths_to_overwrite = [metrics_path, entropy_path, corr_path, marker_path, charge_path, summary_path, latest_path]
    if frustration_path is not None:
        paths_to_overwrite.append(frustration_path)
    for path in paths_to_overwrite:
        assert_overwrite_allowed(path)

    save_dataframe_atomic_csv(metrics_df, metrics_path)
    save_npz_atomic(entropy_path, **observer.entropy_npz_payload(config=run_payload))
    save_npz_atomic(corr_path, **observer.square_correlator_npz_payload(config=run_payload))
    save_npz_atomic(marker_path, **observer.local_chern_marker_npz_payload(config=run_payload))
    save_npz_atomic(charge_path, **observer.local_charge_cell_npz_payload(config=run_payload))
    if frustration_path is not None:
        save_npz_atomic(frustration_path, **observer.measurement_frustration_npz_payload(config=run_payload))

    run_summary = {
        **run_payload,
        'run_dir': str(out_dir),
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'scalar_metrics_relative': rel_to_root(metrics_path, GPU_DATA_ROOT),
        'entropy_y0avg_vs_ay_relative': rel_to_root(entropy_path, GPU_DATA_ROOT),
        'xavg_square_correlator_vs_ry_relative': rel_to_root(corr_path, GPU_DATA_ROOT),
        'local_chern_marker_relative': rel_to_root(marker_path, GPU_DATA_ROOT),
        'local_charge_cell_relative': rel_to_root(charge_path, GPU_DATA_ROOT),
        'measurement_frustration_relative': rel_to_root(frustration_path, GPU_DATA_ROOT),
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'batch_size_auto_info': result.get('batch_size_auto_info'),
        'metric_row_count': int(len(metrics_df)),
        'helper_version': HELPER_VERSION,
        'saved_file_sizes_bytes': {
            'scalar_metrics': file_size_bytes(metrics_path),
            'entropy_y0avg_vs_ay': file_size_bytes(entropy_path),
            'xavg_square_correlator_vs_ry': file_size_bytes(corr_path),
            'local_chern_marker': file_size_bytes(marker_path),
            'local_charge_cell': file_size_bytes(charge_path),
            **({'measurement_frustration': file_size_bytes(frustration_path)} if frustration_path is not None else {}),
        },
        **observer.run_summary_metrics(),
    }
    run_summary['saved_total_bytes'] = int(sum(run_summary['saved_file_sizes_bytes'].values()))
    write_json_atomic(summary_path, run_summary)
    write_json_atomic(
        latest_path,
        {
            'run_summary_path': str(summary_path),
            'run_summary_path_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
            **run_summary,
        },
    )

    return {
        'config_id': config_id(cfg['Nx'], cfg['Ny'], protocol),
        'geometry_key': geometry_key(cfg['Nx'], cfg['Ny']),
        'protocol': protocol,
        'protocol_display': cfg['protocol_display'],
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': cycles,
        'nshell': int(cfg['nshell']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': actual_samples,
        'batch_size_requested': cfg['batch_size'],
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'run_summary_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
        'saved_total_bytes': int(run_summary['saved_total_bytes']),
        'scalar_metrics_bytes': int(run_summary['saved_file_sizes_bytes']['scalar_metrics']),
        'entropy_bytes': int(run_summary['saved_file_sizes_bytes']['entropy_y0avg_vs_ay']),
        'square_corr_bytes': int(run_summary['saved_file_sizes_bytes']['xavg_square_correlator_vs_ry']),
        'local_chern_marker_bytes': int(run_summary['saved_file_sizes_bytes']['local_chern_marker']),
        'local_charge_cell_bytes': int(run_summary['saved_file_sizes_bytes']['local_charge_cell']),
        'measurement_frustration_bytes': int(run_summary['saved_file_sizes_bytes'].get('measurement_frustration', 0)),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }


In [ ]:
all_results = []
campaign_start = time.perf_counter()
for cfg in CONFIGS:
    all_results.append(run_one_config(cfg))
elapsed_s = float(time.perf_counter() - campaign_start)

run_index = pd.DataFrame(all_results).sort_values(['Nx', 'Ny', 'protocol']).reset_index(drop=True)
run_index_path = CAMPAIGN_ROOT / 'run_index.csv'
assert_overwrite_allowed(run_index_path)
save_dataframe_atomic_csv(run_index, run_index_path)

campaign_manifest = {
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'helper_version': HELPER_VERSION,
    'bundle_root': str(BUNDLE_ROOT),
    'project_root': str(PROJECT_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_root(OUTPUT_ROOT, GPU_DATA_ROOT),
    'campaign_root_relative': rel_to_root(CAMPAIGN_ROOT, GPU_DATA_ROOT),
    'runs_root_relative': rel_to_root(RUNS_ROOT, GPU_DATA_ROOT),
    'run_index_relative': rel_to_root(run_index_path, GPU_DATA_ROOT),
    'configs': CONFIGS,
    'results': all_results,
    'requested_samples': REQUESTED_SAMPLES,
    'dtype': DTYPE,
    'init_mode': INIT_MODE,
    'dw_truncation': DW_TRUNCATION,
    'trial_orbitals': TRIAL_ORBITALS,
    'backend': BACKEND,
    'sequence': SEQUENCE,
    'covariance_history_saved': False,
    'observable_collection_mode': 'protocol_specific_streaming_observers',
    'saved_observables_by_protocol': SAVED_OBSERVABLES_BY_PROTOCOL,
    'campaign_outputs': {
        'run_index_relative': rel_to_root(run_index_path, GPU_DATA_ROOT),
    },
    'saved_total_bytes': int(run_index['saved_total_bytes'].sum()),
    'elapsed_seconds': elapsed_s,
    'note': 'postselect=True requests 100 samples but the canonical GPU driver forces samples=1.',
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
assert_overwrite_allowed(campaign_manifest_path)
write_json_atomic(campaign_manifest_path, campaign_manifest)

latest_campaign_path = OUTPUT_ROOT / 'latest_campaign.json'
assert_overwrite_allowed(latest_campaign_path)
write_json_atomic(
    latest_campaign_path,
    {
        'campaign_name': CAMPAIGN_NAME,
        'campaign_id': CAMPAIGN_ID,
        'manifest': rel_to_root(campaign_manifest_path, GPU_DATA_ROOT),
        'campaign_manifest_path': str(campaign_manifest_path),
        'campaign_manifest_path_relative': rel_to_root(campaign_manifest_path, GPU_DATA_ROOT),
    },
)
update_gpu_data_index()

print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] latest campaign pointer: {latest_campaign_path}')
print(f'[done] run index: {run_index_path}')
display(Markdown('### Saved run index'))
display(run_index)


In [ ]:
from IPython.display import Javascript, display

try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
